In [1]:
import numpy as np


# --- Config and Dataset Loading ---
# Set these variables to configure your scene and utility path

scene_name = 'spagna_train0'
dataset_root_dir = "/datasets/vbr_slam"
vbr_utils_root = '/home/bjangley/VPR/mast3r-v2/my_vbr_utils'
gps_path = f"{vbr_utils_root}/global_trajectory/{scene_name}/trajectory.csv"


SEQUENCE_ANCHOR_RANGES = {
"S0kbYi8VQ_WxGBZSOrY3Wg": [[37300, 41500, 50]],
"8YkP2nVvfrlIlRvsfwop8A": [[37300, 41500, 50]],
"n4k0JDQfdeItNMf9OOwCUQ": [
[7781, 7822, 5],
[23861, 23883, 5],
[2531, 2600, 5]
]
}

## Uncomment the one you want to plot and inspect
scene="spagna_train0"
sequence="S0kbYi8VQ_WxGBZSOrY3Wg"


# scene="spagna_train0"
# sequence="n4k0JDQfdeItNMf9OOwCUQ"

# scene="campus_train0"
# sequence="5kb1M1svQmCdlwVYX6iP4Q"



In [2]:
import os
import pandas as pd
import folium


output_root = f"/home/bjangley/VPR/mast3r-v2/results_mapillary/{scene}/{sequence}"
metadata_csv = f"/home/bjangley/VPR/mapillary_utils/vbr_mapillary_downloads/{scene}/{sequence}/metadata.csv"
results_csv = os.path.join(output_root, 'anydepth80.csv')
lidar_csv = os.path.join(output_root, 'lidar.csv')

# === Load ===
df_res  = pd.read_csv(results_csv)
df_lid  = pd.read_csv(lidar_csv)
df_meta = pd.read_csv(metadata_csv)

if df_res.empty:
    raise ValueError(f"[ERROR] Empty results: {results_csv}")
if df_lid.empty:
    raise ValueError(f"[ERROR] Empty LiDAR results: {lidar_csv}")
if df_meta.empty:
    raise ValueError(f"[ERROR] Empty metadata: {metadata_csv}")

# --- Normalize key columns ---
df_res['query_idx']  = df_res['query_idx'].astype(str)
df_res['anchor_idx'] = df_res['anchor_idx'].astype(int)
df_lid['query_idx']  = df_lid['query_idx'].astype(str)
df_lid['anchor_idx'] = df_lid['anchor_idx'].astype(int)
df_meta['id']        = df_meta['id'].astype(str)

# --- Find capture time column in metadata ---
time_candidates = ['captured_at', 'capturedAt', 'capture_time', 'timestamp', 'created_at']
time_col = next((c for c in time_candidates if c in df_meta.columns), None)
if time_col is None:
    raise ValueError(f"Could not find a capture time column in metadata. Tried: {time_candidates}")

# Parse times in metadata
df_meta['capture_time'] = pd.to_datetime(df_meta[time_col], errors='coerce')

# --- Lon/Lat column names in metadata ---
lat_col_candidates = ['lat', 'latitude']
lon_col_candidates = ['long', 'lon', 'longitude']
lat_col = next((c for c in lat_col_candidates if c in df_meta.columns), None)
lon_col = next((c for c in lon_col_candidates if c in df_meta.columns), None)
if lat_col is None or lon_col is None:
    raise ValueError(f"Could not find lat/lon columns in metadata. Tried lat={lat_col_candidates}, lon={lon_col_candidates}")

# --- Keep ONLY rows where a LiDAR localization exists ---
df_lid_small = df_lid[['query_idx', 'anchor_idx', 'lat', 'lon']].rename(
    columns={'lat':'lidar_lat', 'lon':'lidar_lon'}
)
df_merged = pd.merge(
    df_lid_small,
    df_res[['query_idx','anchor_idx','lat','lon','num_inliers']],
    on=['query_idx','anchor_idx'],
    how='inner',
    suffixes=('_lidar','_any')
)

# --- Attach metadata (GPS + capture time) before filtering ---
df_merged = pd.merge(
    df_merged,
    df_meta[['id', lat_col, lon_col, 'capture_time']].rename(
        columns={'id':'query_idx', lat_col:'gps_lat', lon_col:'gps_lon'}
    ),
    on='query_idx',
    how='left'
)

# --- Apply global inlier filter ---
INLIER_MIN = 1000
df_merged['num_inliers'] = pd.to_numeric(df_merged['num_inliers'], errors='coerce')
df_merged = df_merged[df_merged['num_inliers'] >= INLIER_MIN].copy()
if df_merged.empty:
    raise ValueError(f"No results after applying inlier filter ≥ {INLIER_MIN}.")

# --- Drop anything without valid capture time or coords ---
df_merged = df_merged[
    df_merged['capture_time'].notna()
    & df_merged['gps_lat'].notna() & df_merged['gps_lon'].notna()
    & df_merged['lat'].notna() & df_merged['lon'].notna()
    & df_merged['lidar_lat'].notna() & df_merged['lidar_lon'].notna()
].copy()

# --- Sort ---
df_merged.sort_values('capture_time', inplace=True)

# --- Build coordinate sequences (ordered) ---
gps_coords      = list(zip(df_merged['gps_lat'].astype(float),   df_merged['gps_lon'].astype(float)))
anydepth_coords = list(zip(df_merged['lat'].astype(float),       df_merged['lon'].astype(float)))
lidar_coords    = list(zip(df_merged['lidar_lat'].astype(float), df_merged['lidar_lon'].astype(float)))

# Center map at the first GPS (or AnyDepth) point
center_lat, center_lon = (gps_coords[0] if len(gps_coords) else anydepth_coords[0])

# --- Create map ---
tile_url = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}'
attr = "Tiles © Esri — Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community"

m = folium.Map(location=[center_lat, center_lon], zoom_start=18, control_scale=True)
folium.TileLayer(tiles=tile_url, attr=attr, name='Esri World Imagery').add_to(m)

fg_gps      = folium.FeatureGroup(name='Original GPS (LiDAR-subset)', show=True)
fg_anydepth = folium.FeatureGroup(name='AnyDepth Estimation (LiDAR-subset)', show=True)
fg_lidar    = folium.FeatureGroup(name='LiDAR Estimation', show=True)
fg_anchors  = folium.FeatureGroup(name='Anchor Positions', show=True)

def add_path_with_points(fg, coords, color, label):
    if len(coords) >= 2:
        folium.PolyLine(coords, color=color, weight=1, opacity=0.9).add_to(fg)
        # Start/End markers
        folium.CircleMarker(coords[0],  radius=3, color=color, fill=True, fill_color=color,
                            popup=f"{label} • start").add_to(fg)
        folium.CircleMarker(coords[-1], radius=3, color=color, fill=True, fill_color=color,
                            popup=f"{label} • end").add_to(fg)
    # Drop a tiny point at each sample for inspection
    for i, (lat, lon) in enumerate(coords):
        folium.CircleMarker((lat, lon), radius=1, color=color, fill=True, fill_color=color,
                            fill_opacity=0.8, popup=f"{label} • idx={i}").add_to(fg)

# Add layers
add_path_with_points(fg_gps, gps_coords, '#00ff00', 'GPS')
add_path_with_points(fg_anydepth, anydepth_coords, 'red',   'AnyDepth')
add_path_with_points(fg_lidar, lidar_coords, '#00bfff', 'LiDAR')

# --- Load anchor trajectory (factory GPS) ---
df_gps = pd.read_csv(gps_path)
if not {'latitude','longitude'}.issubset(df_gps.columns):
    raise ValueError("trajectory.csv must have latitude/longitude columns")

anchor_coords = []
for idx in df_merged['anchor_idx'].unique():
    if idx < len(df_gps):
        anchor_coords.append((df_gps['latitude'].iloc[idx], df_gps['longitude'].iloc[idx]))

for i, (lat, lon) in enumerate(anchor_coords):
    folium.CircleMarker(
        (lat, lon),
        radius=1,
        color='yellow',
        fill=True,
        fill_color='yellow',
        popup=f"Anchor idx={i}"
    ).add_to(fg_anchors)

# Add all layers
fg_gps.add_to(m)
fg_anydepth.add_to(m)
fg_lidar.add_to(m)
fg_anchors.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

# Legend
legend_html = f'''
<div style="
    position: fixed; bottom: 50px; left: 50px; width: 320px;
    background-color: white; border: 2px solid grey; z-index: 9999;
    font-size: 14px; padding: 10px; border-radius: 8px;
    box-shadow: 2px 2px 8px rgba(0,0,0,0.3);">
  <b>Legend (filtered: inliers ≥ {INLIER_MIN})</b><br>
  <span style="display:inline-block;width:14px;height:14px;background:green;margin-right:6px;"></span> GPS (polyline + points)<br>
  <span style="display:inline-block;width:14px;height:14px;background:red;margin-right:6px;"></span> AnyDepth (polyline + points)<br>
  <span style="display:inline-block;width:14px;height:14px;background:blue;margin-right:6px;"></span> LiDAR (polyline + points)<br>
  <span style="display:inline-block;width:14px;height:14px;background:yellow;margin-right:6px;"></span> Anchor positions
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Jupyter display
m


In [3]:
import os
import pandas as pd
import folium
from itertools import combinations

from folium.features import DivIcon

def add_numbered_marker(fg, location, number, color='#ff0000', popup_text=None):
    html = f'''
    <div style="position: relative; width: 30px; height: 42px;">

      <div style="
          background-color: {color};
          color: white;
          border-radius: 50%;
          width: 30px;
          height: 30px;
          text-align: center;
          font-weight: bold;
          font-size: 16px;
          line-height: 30px;
          box-shadow: 0 0 2px #555;
          position: absolute;
          top: 0;
          left: 0;
          z-index: 10;
      ">{number}</div>

      <div style="
          width: 0;
          height: 0;
          border-left: 15px solid transparent;
          border-right: 15px solid transparent;
          border-top: 12px solid {color};
          position: absolute;
          top: 28px;
          left: 0;
          z-index: 5;
      "></div>

    </div>
    '''
    folium.Marker(
        location=location,
        icon=DivIcon(
            icon_size=(30, 42),
            icon_anchor=(15, 42),
            html=html
        ),
        popup=popup_text
    ).add_to(fg)


# # Paths (customize these)
output_root = f"/home/bjangley/VPR/mast3r-v2/results_mapillary/{scene}/{sequence}"
metadata_csv = f"/home/bjangley/VPR/mapillary_utils/vbr_mapillary_downloads/{scene}/{sequence}/metadata.csv"
results_csv = os.path.join(output_root, 'anydepth80.csv')
lidar_csv = os.path.join(output_root, 'lidar.csv')
vbr_utils_root = '/home/bjangley/VPR/mast3r-v2/my_vbr_utils'
gps_path = f"{vbr_utils_root}/global_trajectory/{scene}/trajectory.csv"
# === Load ===
df_res  = pd.read_csv(results_csv)
df_lid  = pd.read_csv(lidar_csv)
df_meta = pd.read_csv(metadata_csv)

# Normalize keys
df_res['query_idx']  = df_res['query_idx'].astype(str)
df_res['anchor_idx'] = df_res['anchor_idx'].astype(int)
df_lid['query_idx']  = df_lid['query_idx'].astype(str)
df_lid['anchor_idx'] = df_lid['anchor_idx'].astype(int)
df_meta['id']        = df_meta['id'].astype(str)

# Capture time
time_candidates = ['captured_at','capturedAt','capture_time','timestamp','created_at']
time_col = next((c for c in time_candidates if c in df_meta.columns), None)
df_meta['capture_time'] = pd.to_datetime(df_meta[time_col], errors='coerce')

# Lat/Lon
lat_col = next(c for c in ['lat','latitude'] if c in df_meta.columns)
lon_col = next(c for c in ['long','lon','longitude'] if c in df_meta.columns)

# Merge lidar + results
df_lid_small = df_lid[['query_idx','anchor_idx','lat','lon']].rename(
    columns={'lat':'lidar_lat','lon':'lidar_lon'}
)
df_merged = pd.merge(
    df_lid_small,
    df_res[['query_idx','anchor_idx','lat','lon','num_inliers']],
    on=['query_idx','anchor_idx'],
    how='inner'
)

# Add metadata
df_merged = pd.merge(
    df_merged,
    df_meta[['id',lat_col,lon_col,'capture_time']].rename(
        columns={'id':'query_idx',lat_col:'gps_lat',lon_col:'gps_lon'}
    ),
    on='query_idx',
    how='left'
)

# Filter for inliers
INLIER_MIN = 700
df_merged['num_inliers'] = pd.to_numeric(df_merged['num_inliers'], errors='coerce')
df_merged = df_merged[df_merged['num_inliers'] >= INLIER_MIN].dropna()

# --- Additional robust filtering for valid coordinates ---
def is_valid_coord(lat, lon):
    return (
        pd.notnull(lat) and pd.notnull(lon) and
        lat != 0 and lon != 0 and
        -90 < lat < 90 and -180 < lon < 180
    )

df_merged = df_merged[
    df_merged.apply(
        lambda row: is_valid_coord(row['gps_lat'], row['gps_lon']) and
                    is_valid_coord(row['lat'], row['lon']) and
                    is_valid_coord(row['lidar_lat'], row['lidar_lon']),
        axis=1
    )
]

df_merged.sort_values('capture_time', inplace=True)

# Anchor GPS trajectory
df_gps = pd.read_csv(gps_path)

# === Build coordinate sequences (ordered) ===
gps_coords      = list(zip(df_merged['gps_lat'],   df_merged['gps_lon']))
anydepth_coords = list(zip(df_merged['lat'],       df_merged['lon']))
lidar_coords    = list(zip(df_merged['lidar_lat'], df_merged['lidar_lon']))

# --- Base map ---
tile_url = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}'
attr = "Tiles © Esri, Maxar, Earthstar Geographics, GIS User Community"
center_lat, center_lon = gps_coords[0]
m = folium.Map(location=[center_lat, center_lon], zoom_start=18, control_scale=True,
               tiles=tile_url, attr=attr)
fg_gps      = folium.FeatureGroup(name='GPS (LiDAR-subset)', show=True)
fg_anydepth = folium.FeatureGroup(name='AnyDepth', show=True)
fg_lidar    = folium.FeatureGroup(name='LiDAR', show=True)
fg_anchors  = folium.FeatureGroup(name='Anchors', show=True)
fg_pairs    = folium.FeatureGroup(name='Specific Pairs', show=True)
fg_queries    = folium.FeatureGroup(name='Queries', show=True)

def add_path_with_points(fg, df, lat_col, lon_col, color, label):
    coords = list(zip(df[lat_col], df[lon_col]))
    if len(coords) >= 2:
        folium.PolyLine(coords, color=color, weight=2, opacity=0.9).add_to(fg)
    for _, row in df.iterrows():
        folium.CircleMarker(
            (row[lat_col], row[lon_col]),
            radius=1, color=color, fill=True, fill_color=color,
            popup=f"{label} • Query {row['query_idx']}"
        ).add_to(fg)

# Draw full trajectories
add_path_with_points(fg_gps,      df_merged, 'gps_lat',   'gps_lon', '#00ff00', 'GPS')
add_path_with_points(fg_anydepth, df_merged, 'lat',       'lon',     'red',   'AnyDepth')
add_path_with_points(fg_lidar,    df_merged, 'lidar_lat', 'lidar_lon','#00bfff', 'LiDAR')

# Anchors
# for anchor_idx in df_merged['anchor_idx'].unique():
#     if anchor_idx < len(df_gps):
#         lat_a, lon_a = df_gps['latitude'].iloc[anchor_idx], df_gps['longitude'].iloc[anchor_idx]
#         queries = df_merged[df_merged['anchor_idx']==anchor_idx]['query_idx'].unique()
#         popup_text = f"Anchor {anchor_idx}<br>Queries: {', '.join(queries)}"
#         folium.CircleMarker((lat_a, lon_a), radius=1, color='yellow', fill=True,
#                             fill_color='yellow', popup=popup_text).add_to(fg_anchors)

# Extract unique anchor indices present in df_merged and sort them
selected_anchors = sorted(df_merged['anchor_idx'].unique())

# Get the lat/lon for those anchors from df_gps
selected_anchor_coords = [(df_gps['latitude'].iloc[idx], df_gps['longitude'].iloc[idx]) for idx in selected_anchors]

# Plot a polyline only for the selected anchor points
if len(selected_anchor_coords) >= 2:
    folium.PolyLine(selected_anchor_coords, color='yellow', weight=3, opacity=1, popup='Selected Anchor Path').add_to(fg_anchors)

def highlight_query(anchor_idx, query_idx, number):
    row = df_merged[(df_merged['anchor_idx'] == anchor_idx) & (df_merged['query_idx'] == str(query_idx))]
    if row.empty:
        print(f"Pair Anchor {anchor_idx}, Query {query_idx} not found.")
        return

    row = row.iloc[0]

    # GPS query point with numbered marker
    folium.CircleMarker(
        location=(row['gps_lat'], row['gps_lon']),
        radius=4,
        color="black",
        fill=True,
        fill_color="white",
        fill_opacity=0.8,
        popup=f"GPS • Query {row['query_idx']}"
    ).add_to(fg_queries)
    add_numbered_marker(fg_queries, (row['gps_lat'], row['gps_lon']), number=number, color='green', popup_text=f"GPS • Query {row['query_idx']}")



def highlight_result(anchor_idx, query_idx, number):
    row = df_merged[(df_merged['anchor_idx'] == anchor_idx) & (df_merged['query_idx'] == str(query_idx))]
    if row.empty:
        print(f"Pair Anchor {anchor_idx}, Query {query_idx} not found.")
        return

    row = row.iloc[0]

    # GPS query point with numbered marker
    folium.CircleMarker(
        location=(row['gps_lat'], row['gps_lon']),
        radius=4,
        color="black",
        fill=True,
        fill_color="white",
        fill_opacity=0.8,
        popup=f"GPS • Query {row['query_idx']}"
    ).add_to(fg_pairs)
    add_numbered_marker(fg_pairs, (row['gps_lat'], row['gps_lon']), number=number, color='green', popup_text=f"GPS • Query {row['query_idx']}")

    # AnyDepth query point with numbered marker
    folium.CircleMarker(
        location=(row['lat'], row['lon']),
        radius=6,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.8,
        popup=f"AnyDepth • Query {row['query_idx']}"
    ).add_to(fg_pairs)
    # add_numbered_marker(fg_pairs, (row['lat'], row['lon']), number=number, color='red', popup_text=f"AnyDepth • Query {row['query_idx']}")

    # LiDAR query point with numbered marker
    folium.CircleMarker(
        location=(row['lidar_lat'], row['lidar_lon']),
        radius=4,
        color="#00bfff",
        fill=True,
        fill_color="#00bfff",
        fill_opacity=0.8,
        popup=f"LiDAR • Query {row['query_idx']}"
    ).add_to(fg_pairs)
    # add_numbered_marker(fg_pairs, (row['lidar_lat'], row['lidar_lon']), number=number, color='#00bfff', popup_text=f"LiDAR • Query {row['query_idx']}")

    # Anchor point without numbered marker, just circle
    if anchor_idx < len(df_gps):
        lat_a, lon_a = df_gps['latitude'].iloc[anchor_idx], df_gps['longitude'].iloc[anchor_idx]
        folium.CircleMarker(
            location=(lat_a, lon_a),
            radius=4,
            color="yellow",
            fill=True,
            fill_color="yellow",
            fill_opacity=0.8,
            popup=f"Anchor {anchor_idx} • Query {query_idx}"
        ).add_to(fg_pairs)

    # Draw lines between all query markers for this pair (optional)
    points = [
        (row['gps_lat'], row['gps_lon']),
        (row['lat'], row['lon']),
        (row['lidar_lat'], row['lidar_lon'])
    ]
    if anchor_idx < len(df_gps):
        points.append((lat_a, lon_a))
    for p1, p2 in combinations(points, 2):
        folium.PolyLine([p1, p2], color="white", weight=1, opacity=1).add_to(fg_pairs)



# Example: highlight one pair

highlight_query(anchor_idx=39050, query_idx="138940178222730", number=2)
highlight_query(anchor_idx=39850, query_idx="303730374594771", number=3)
highlight_query(anchor_idx=38400, query_idx="762142218000244", number=1)
highlight_query(anchor_idx=40850, query_idx="214786096833734", number=4)


highlight_query(anchor_idx=7811, query_idx="306713820894516",number="Q")

highlight_result(anchor_idx=39050, query_idx="138940178222730", number=2)
highlight_result(anchor_idx=39850, query_idx="303730374594771", number=3)
highlight_result(anchor_idx=38400, query_idx="762142218000244", number=1)
highlight_result(anchor_idx=40850, query_idx="214786096833734", number=4)


highlight_result(anchor_idx=7811, query_idx="306713820894516",number="Q")

#campus_train0
highlight_query(anchor_idx=11230, query_idx="769642790366382", number=1)
highlight_query(anchor_idx=11350, query_idx="1109672756221448", number=2)
highlight_query(anchor_idx=11460, query_idx="856560084943113", number=3)
# # highlight_pair(anchor_idx=11440, query_idx="323224029233485", number=4)

highlight_result(anchor_idx=11230, query_idx="769642790366382", number=1)
highlight_result(anchor_idx=11350, query_idx="1109672756221448", number=2)
highlight_result(anchor_idx=11460, query_idx="856560084943113", number=3)
# highlight_pair(anchor_idx=11440, query_idx="323224029233485", number=4)

# Add layers
fg_gps.add_to(m)
fg_anydepth.add_to(m)
fg_lidar.add_to(m)
fg_anchors.add_to(m)
fg_pairs.add_to(m)
fg_queries.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

legend_html = '''
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 150px;
    background-color: white;
    border: 2px solid gray;
    z-index: 9999;
    font-size: 10px;
    padding: 10px;
    box-shadow: 3px 3px 6px rgba(0,0,0,0.3);
">
<strong>Legend</strong><br>
<span style="background:#00ff00; width:15px; height:15px; display:inline-block; margin-right:8px;"></span> Mapillary GPS<br>
<span style="background:yellow; width:15px; height:15px; display:inline-block; margin-right:8px;"></span> Anchors<br>
<span style="background:red; width:15px; height:15px; display:inline-block; margin-right:8px;"></span> Depth Anything V2<br>
<span style="background:#00bfff; width:15px; height:15px; display:inline-block; margin-right:8px;"></span>Oracle<br>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))


# Display map (in notebook; or save with m.save('your_map.html'))
m
# m.save('your_map.html')

Pair Anchor 7811, Query 306713820894516 not found.
Pair Anchor 7811, Query 306713820894516 not found.
Pair Anchor 11230, Query 769642790366382 not found.
Pair Anchor 11350, Query 1109672756221448 not found.
Pair Anchor 11460, Query 856560084943113 not found.
Pair Anchor 11230, Query 769642790366382 not found.
Pair Anchor 11350, Query 1109672756221448 not found.
Pair Anchor 11460, Query 856560084943113 not found.
